# 🔏 Swin Transformer Invisible Watermarking
**Architecture:** Swin Transformer Encoder + Frequency Enhancement + CNN Decoder  
**Dataset:** DIV2K (128×128 patches)  
**Message:** 64-bit payload with optional BCH error correction  
**Checkpoints:** Best-ever model + last checkpoint saved to Google Drive (2 files only)  
**Logging:** Visual collages saved to Drive every N batches  

## ⚙️ Cell 2 — Imports & Device

In [2]:
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


/home/shrine/Minor/RoWSFormer/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


## 💾 Cell 3 — Set dataset config and checkpoints config

In [3]:
PROJECT_DIR = os.getcwd()
if os.path.basename(PROJECT_DIR) == 'notebooks':
    PROJECT_DIR = os.path.dirname(PROJECT_DIR)

OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs')
CKPT_DIR   = os.path.join(OUTPUT_DIR, 'checkpoints')
LOG_DIR    = os.path.join(OUTPUT_DIR, 'collages')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

BEST_CKPT = os.path.join(CKPT_DIR, 'best_model.pth')
LAST_CKPT = os.path.join(CKPT_DIR, 'last_checkpoint.pth')
RUNS_LOG  = os.path.join(CKPT_DIR, 'runs_log.json')   # ← only addition

print(f'Output dirs ready → {OUTPUT_DIR}')

Output dirs ready → /home/shrine/Minor/RoWSFormer/outputs


## 🔧 Cell 4 — Configuration

In [4]:
from datetime import datetime
import json 

# ─── Toggle BCH Error Correction ─────────────────────────────────
USE_BCH = False
# ─────────────────────────────────────────────────────────────────

DATA_BITS = 64

if USE_BCH:
    import bchlib
    BCH_POLYNOMIAL = 137
    BCH_BITS       = 10
    _bch_tmp       = bchlib.BCH(BCH_BITS, prim_poly=BCH_POLYNOMIAL)
    _ecc_tmp       = _bch_tmp.encode(bytes(8))
    MESSAGE_LENGTH = (8 + len(_ecc_tmp)) * 8
    print(f'BCH ON  →  data bits: {DATA_BITS}  |  codeword bits: {MESSAGE_LENGTH}')
else:
    MESSAGE_LENGTH = DATA_BITS
    print(f'BCH OFF →  message bits: {MESSAGE_LENGTH}')

# ── Hyperparameters ───────────────────────────────────────────────
IMG_SIZE   = 128
BATCH_SIZE = 16
EPOCHS     = 100
LR         = 1e-4
LR_MIN     = 1e-6

LAMBDA_IMG  = 0.5
LAMBDA_MSG  = 8.0
LAMBDA_ADV  = 0.001
LAMBDA_FREQ = 0.05

LOG_EVERY_N_BATCHES = 200
PRINT_EVERY         = 20

print('Config loaded ✅')

# ── Register this run in the log ──────────────────────────────────
# NOTE: EXP_DIR must exist (defined in Cell 3) before this runs
_cfg = {
    "noise": "geometric",
    "lr": LR, "lr_min": LR_MIN, "epochs": EPOCHS,
    "lambda_msg": LAMBDA_MSG, "lambda_img": LAMBDA_IMG,
    "lambda_adv": LAMBDA_ADV, "lambda_freq": LAMBDA_FREQ,
    "batch_size": BATCH_SIZE, "img_size": IMG_SIZE,
    "use_bch": USE_BCH,
}

# ── Register this run in the log ──────────────────────────────────
try:
    with open(RUNS_LOG) as f:
        _log = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    _log = []   # handles missing file OR empty/corrupt fi

_run_id = len(_log) + 1
_log.append({
    "run_id"    : _run_id,
    "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M"),
    "config"    : _cfg,
    "result"    : None   # filled at end of training
})
with open(RUNS_LOG, 'w') as f:
    json.dump(_log, f, indent=2)
print(f'Run #{_run_id} registered ✅')

BCH OFF →  message bits: 64
Config loaded ✅
Run #25 registered ✅


## BCH encoders and helpers

In [5]:
if USE_BCH:
    import bchlib
    _bch = bchlib.BCH(BCH_BITS, prim_poly=BCH_POLYNOMIAL)

    def bch_encode_bits(bit_tensor):
        """(B, DATA_BITS) float → (B, MESSAGE_LENGTH) float"""
        B, out = bit_tensor.shape[0], []
        for b in range(B):
            bits = bit_tensor[b].detach().cpu().numpy().astype(np.uint8)
            data_bytes = np.packbits(bits).tobytes()
            ecc = _bch.encode(data_bytes)
            full_bits = np.unpackbits(np.frombuffer(data_bytes + ecc, dtype=np.uint8))
            out.append(full_bits[:MESSAGE_LENGTH])
        return torch.tensor(np.stack(out), dtype=torch.float32).to(bit_tensor.device)

    def bch_decode_bits(codeword_tensor):
        """(B, MESSAGE_LENGTH) float → (B, DATA_BITS) float"""
        B, out = codeword_tensor.shape[0], []
        for b in range(B):
            bits = (codeword_tensor[b] > 0.5).cpu().numpy().astype(np.uint8)
            pad  = (8 - len(bits) % 8) % 8
            bits = np.concatenate([bits, np.zeros(pad, dtype=np.uint8)])
            raw  = np.packbits(bits).tobytes()
            data_b, ecc_b = raw[:8], raw[8:8+len(_bch.encode(raw[:8]))]
            try:
                _, data_corr, _ = _bch.decode(data_b, ecc_b)
            except Exception:
                data_corr = data_b
            dec = np.unpackbits(np.frombuffer(data_corr, dtype=np.uint8))[:DATA_BITS]
            out.append(dec.astype(np.float32))
        return torch.tensor(np.stack(out), dtype=torch.float32).to(codeword_tensor.device)

    print(f'BCH helpers ready. Codeword: {MESSAGE_LENGTH} bits')
else:
    bch_encode_bits = bch_decode_bits = None
    print('BCH disabled — using raw 64-bit messages')


BCH disabled — using raw 64-bit messages


## 🗃️ Cell 7 — DIV2K Dataset & DataLoader

In [6]:
DIV2K_ROOT   = os.path.abspath('../data/DIV2K/DIV2K_train_HR/DIV2K_train_HR')

class DIV2KDataset(Dataset):
    """Random 128x128 crops from DIV2K HR images."""
    def __init__(self, root, patch_size=128, augment=True):
        self.files = sorted([
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.patch_size = patch_size
        self.augment    = augment
        self.to_tensor  = T.ToTensor()

    def __len__(self):
        return len(self.files) * 20

    def __getitem__(self, idx):
        img = Image.open(self.files[idx % len(self.files)]).convert('RGB')
        w, h = img.size
        p = self.patch_size
        if w < p or h < p:
            img = img.resize((max(w,p), max(h,p)), Image.BICUBIC)
            w, h = img.size
        x0 = random.randint(0, w-p)
        y0 = random.randint(0, h-p)
        patch = img.crop((x0, y0, x0+p, y0+p))
        if self.augment:
            if random.random() > 0.5: patch = TF.hflip(patch)
            if random.random() > 0.5: patch = TF.vflip(patch)
        return self.to_tensor(patch)



train_ds     = DIV2KDataset(DIV2K_ROOT, patch_size=IMG_SIZE, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0, pin_memory=True)
print(f'Dataset: {len(train_ds.files)} HR images  →  {len(train_ds)} patches')
print(f'Loader : {len(train_loader)} batches/epoch')


Dataset: 800 HR images  →  16000 patches
Loader : 1000 batches/epoch


## 🌊 Cell 8 — Frequency Enhancement Module

In [7]:
class FrequencyEnhancement(nn.Module):
    """
    Decomposes feature maps into low/high frequency bands,
    processes each with a learnable gate, then merges.
    Encourages the watermark to live in perceptually-safe mid-frequencies.
    """
    def __init__(self, channels):
        super().__init__()
        self.low  = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels), nn.GELU()
        )
        self.high = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels), nn.GELU()
        )
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels), nn.Sigmoid()
        )
        self.proj = nn.Conv2d(channels*2, channels, 1)

    def forward(self, x):
        lf  = self.low(x)
        hf  = self.high(x - lf)
        g   = self.gate(x).unsqueeze(-1).unsqueeze(-1)
        out = self.proj(torch.cat([lf*g, hf*(1-g)], dim=1))
        return out + x   # residual

print('FrequencyEnhancement module defined ✅')


FrequencyEnhancement module defined ✅


## 🧠 Cell 9 — Swin Transformer Encoder

In [8]:
class MessageProjection(nn.Module):
    """Projects flat message bits into a spatial feature volume."""
    def __init__(self, msg_len, channels, spatial):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(msg_len, channels * spatial * spatial),
            nn.Unflatten(1, (channels, spatial, spatial))
        )
    def forward(self, msg):
        return self.net(msg)


class SwinEncoder(nn.Module):
    """
    Swin Transformer-based watermark encoder.

    Pipeline:
      1. Swin-T backbone extracts multi-scale features from cover image.
      2. Message bits are projected into spatial volumes at each scale.
      3. Image features + message volumes are fused at 4 scales.
      4. FrequencyEnhancement steers perturbation to mid-frequencies.
      5. Hierarchical upsampling reconstructs full-resolution residual.
      6. Residual is added to cover (constrained by learnable scale).
    """
    FUSION_C = 128

    def __init__(self, msg_len=MESSAGE_LENGTH, img_size=128):
        super().__init__()
        C = self.FUSION_C

        # Swin-Tiny backbone (pretrained, img_size adaptable)
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=True, img_size=img_size,
            features_only=True, out_indices=(0,1,2,3)
        )

        swin_chs   = [96, 192, 384, 768]
        swin_spats = [img_size//4, img_size//8, img_size//16, img_size//32]

        self.reduce  = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, C, 1), nn.BatchNorm2d(C), nn.GELU())
            for c in swin_chs
        ])
        # self.msg_proj = nn.ModuleList([
        #     MessageProjection(msg_len, C, s) for s in swin_spats
        # ])
        self.fuse = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(C + msg_len, C, 3, padding=1),
                nn.BatchNorm2d(C), nn.GELU(),
                FrequencyEnhancement(C)
            ) for _ in range(4)
        ])

        def up_block(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, stride=2, padding=1),
                nn.BatchNorm2d(out_c), nn.GELU()
            )

        self.up4 = up_block(C, C)
        self.up3 = up_block(C*2, C)
        self.up2 = up_block(C*2, C)
        self.up1 = up_block(C*2, C)

        self.refine = nn.Sequential(
            FrequencyEnhancement(C),
            nn.Conv2d(C, C, 3, padding=1), nn.BatchNorm2d(C), nn.GELU(),
            nn.Conv2d(C, 3, 1), nn.Tanh()
        )
        self.residual_scale = nn.Parameter(torch.tensor(0.05))

    def forward(self, cover, msg):
        feats = self.swin(cover)
        # Normalise to NCHW (some timm versions return NHWC)
        feats = [
            f.permute(0,3,1,2).contiguous()
            if (f.dim()==4 and f.shape[1]!=f.shape[-1]) else f
            for f in feats
        ]
        red = [self.reduce[i](feats[i]) for i in range(4)]

        fused = []
        for i in range(4):
            h, w = red[i].shape[2:]
            # simple copy: [B, msg_len] → [B, msg_len, 1, 1] → [B, msg_len, H, W]
            mv = msg.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, h, w)
            fused.append(self.fuse[i](torch.cat([red[i], mv], dim=1)))

        x = self.up4(fused[3])
        x = self.up3(torch.cat([x, fused[2]], dim=1))
        x = self.up2(torch.cat([x, fused[1]], dim=1))
        x = self.up1(torch.cat([x, fused[0]], dim=1))

        if x.shape[2:] != cover.shape[2:]:
            x = F.interpolate(x, size=cover.shape[2:],
                              mode='bilinear', align_corners=False)

        residual = self.refine(x)
        scale    = torch.sigmoid(self.residual_scale) * 0.1
        return torch.clamp(cover + residual * scale, 0.0, 1.0)


encoder = SwinEncoder(msg_len=MESSAGE_LENGTH, img_size=IMG_SIZE).to(device)
print(f'Encoder params: {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}')


Encoder params: 30,996,478


/home/shrine/Minor/RoWSFormer/venv/lib64/python3.14/site-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/home/shrine/Minor/RoWSFormer/venv/lib64/python3.14/site-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  numerator += self.va

## 🔍 Cell 10 — CNN Decoder

In [9]:
class ConvDecoder(nn.Module):
    """
    Strided CNN decoder with FrequencyEnhancement blocks.
    Predicts soft bit probabilities from the watermarked image.
    """
    def __init__(self, msg_len=MESSAGE_LENGTH):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),
            FrequencyEnhancement(64),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.GELU(),
            FrequencyEnhancement(128),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            FrequencyEnhancement(256),
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.BatchNorm2d(512), nn.GELU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
        self.head = nn.Sequential(
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, msg_len), nn.Sigmoid()
        )

    def forward(self, x):
        return self.head(self.body(x))


decoder = ConvDecoder(msg_len=MESSAGE_LENGTH).to(device)
print(f'Decoder params: {sum(p.numel() for p in decoder.parameters() if p.requires_grad):,}')


Decoder params: 2,143,296


## 🕵️ Cell 11 — PatchGAN Discriminator

In [10]:
class PatchDiscriminator(nn.Module):
    """PatchGAN — classifies real vs watermarked at patch level."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 1, 4, padding=1),
        )
    def forward(self, x):
        return self.net(x)


discriminator = PatchDiscriminator().to(device)
print(f'Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}')


Discriminator params: 663,745


# Noise Layer


In [11]:
import importlib.util, sys

# ── Add DiffJPEG folder so internal imports (modules, utils) work ──
DIFFJPEG_DIR = '/home/shrine/Minor/RoWSFormer/DiffJPEG'
if DIFFJPEG_DIR not in sys.path:
    sys.path.insert(0, DIFFJPEG_DIR)

# ── Load DiffJPEG ─────────────────────────────────────────────────
spec = importlib.util.spec_from_file_location(
    "DiffJPEG",
    f"{DIFFJPEG_DIR}/DiffJPEG.py"
)
mod = importlib.util.module_from_spec(spec)
sys.modules["DiffJPEG"] = mod
spec.loader.exec_module(mod)
DiffJPEG = mod.DiffJPEG

# ── Differentiable geometric helpers (PyTorch) ────────────────────
def torch_rotate(x, angle_deg):
    angle  = torch.tensor(angle_deg * 3.14159 / 180.0)
    cos_a, sin_a = torch.cos(angle), torch.sin(angle)
    theta  = torch.tensor([[cos_a, -sin_a, 0],
                           [sin_a,  cos_a, 0]], dtype=torch.float32)
    theta  = theta.unsqueeze(0).expand(x.size(0), -1, -1).to(x.device)
    grid   = F.affine_grid(theta, x.size(), align_corners=False)
    return F.grid_sample(x, grid, align_corners=False, padding_mode='reflection')

def torch_scale(x, scale):
    """Scale image — r=0.7 to 1.5 safe range per paper Table IX."""
    theta  = torch.tensor([[scale, 0, 0],
                           [0, scale, 0]], dtype=torch.float32)
    theta  = theta.unsqueeze(0).expand(x.size(0), -1, -1).to(x.device)
    grid   = F.affine_grid(theta, x.size(), align_corners=False)
    return F.grid_sample(x, grid, align_corners=False, padding_mode='reflection')

def torch_translate(x, dx, dy):
    """dx, dy in [-1, 1] normalized coordinates."""
    theta  = torch.tensor([[1, 0, dx],
                           [0, 1, dy]], dtype=torch.float32)
    theta  = theta.unsqueeze(0).expand(x.size(0), -1, -1).to(x.device)
    grid   = F.affine_grid(theta, x.size(), align_corners=False)
    return F.grid_sample(x, grid, align_corners=False, padding_mode='reflection')

def torch_cropout(x, crop_ratio):
    """Zero out a random patch — r=0.1 to 0.3 safe range per paper Table VI."""
    B, C, H, W = x.shape
    out  = x.clone()
    ch   = int(H * crop_ratio)
    cw   = int(W * crop_ratio)
    top  = random.randint(0, H - ch)
    left = random.randint(0, W - cw)
    out[:, :, top:top+ch, left:left+cw] = 0.0
    return out


class NoiseLayer(nn.Module):
    """
    Applies random distortions during training to build robustness.

    Photometric  : Gaussian noise | Color jitter | Gaussian blur | DiffJPEG
    Geometric    : Rotation | Scale | Cropout | Translation  (all differentiable)
    Identity     : 30% pass-through for stability

    Conservative ranges based on paper tables:
      Rotation  : ±15°           (Table VIII — best accuracy zone)
      Scale     : 0.7 to 1.5     (Table IX   — safe range)
      Cropout   : r=0.1 to 0.3   (Table VI   — safe range)
      Translate : up to 5% shift
    """
    def __init__(self):
        super().__init__()
        self.diff_jpeg_75 = DiffJPEG(height=IMG_SIZE, width=IMG_SIZE, differentiable=True, quality=75)
        self.diff_jpeg_80 = DiffJPEG(height=IMG_SIZE, width=IMG_SIZE, differentiable=True, quality=80)
        self.diff_jpeg_85 = DiffJPEG(height=IMG_SIZE, width=IMG_SIZE, differentiable=True, quality=85)

    def forward(self, x):
        if not self.training:
            return x

        r = random.random()

        # ── Photometric (10% each) ────────────────────────────────
        # if r < 0.10:
        #     return torch.clamp(x + 0.03 * torch.randn_like(x), 0, 1)
        # elif r < 0.20:
        #     return T.ColorJitter(brightness=0.1, contrast=0.1)(x)
        # elif r < 0.30:
        #     return T.GaussianBlur(3, sigma=(0.1, 1.0))(x)

        # # ── DiffJPEG (20%) — differentiable ───────────────────────
        # elif r < 0.50:
        #     return random.choice([self.diff_jpeg_75, self.diff_jpeg_80, self.diff_jpeg_85])(x)

        # ── Geometric (20%) — conservative ranges from paper ──────
        if r < 0.2:
            # Rotation: ±15° only — avoids ±30° accuracy drop
            angle = random.choice([-15, -10, -5, 5, 10, 15])
            return torch_rotate(x, angle)
        elif r < 0.4:
            # Scale: 0.7–1.5 safe range
            scale = random.uniform(0.7, 1.5)
            return torch_scale(x, scale)
        elif r < 0.6:
            # Cropout: 10–30% patch size
            crop_ratio = random.uniform(0.1, 0.3)
            return torch_cropout(x, crop_ratio)
        elif r < 0.8:
            # Translate: up to 5% shift
            dx = random.uniform(-0.05, 0.05)
            dy = random.uniform(-0.05, 0.05)
            return torch_translate(x, dx, dy)

        # ── Identity (20%) ────────────────────────────────────────
        else:
            return x


noise_layer = NoiseLayer().to(device)
print('Noise layer ready ✅')

Noise layer ready ✅


## 📐 Cell 13 — Loss Functions & Metrics

In [12]:
def ssim_loss(x, y, w=11, C1=1e-4, C2=9e-4):
    """Differentiable 1-SSIM loss."""
    mu_x = F.avg_pool2d(x, w, 1, w//2)
    mu_y = F.avg_pool2d(y, w, 1, w//2)
    sx   = F.avg_pool2d(x*x, w, 1, w//2) - mu_x**2
    sy   = F.avg_pool2d(y*y, w, 1, w//2) - mu_y**2
    sxy  = F.avg_pool2d(x*y, w, 1, w//2) - mu_x*mu_y
    ssim = ((2*mu_x*mu_y+C1)*(2*sxy+C2))/((mu_x**2+mu_y**2+C1)*(sx+sy+C2))
    return 1 - ssim.mean()


def freq_loss(cover, encoded):
    """Penalise high-frequency differences (Laplacian)."""
    k = torch.tensor([[[[0,-1,0],[-1,4,-1],[0,-1,0]]]],
                     dtype=torch.float32, device=cover.device).expand(3,1,3,3)
    return F.mse_loss(F.conv2d(encoded, k, padding=1, groups=3),
                      F.conv2d(cover,   k, padding=1, groups=3))


def bit_accuracy(pred, target):
    return ((pred > 0.5).float() == target).float().mean().item()


def psnr(cover, encoded):
    mse = F.mse_loss(cover, encoded).item()
    return float('inf') if mse == 0 else 10*math.log10(1.0/mse)


bce_loss = nn.BCELoss()
mse_loss = nn.MSELoss()
adv_bce  = nn.BCEWithLogitsLoss()
print('Loss functions ready ✅')


Loss functions ready ✅


## 🚀 Cell 14 — Optimisers & LR Schedulers

In [13]:
opt_enc_dec = optim.AdamW(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=LR, weight_decay=1e-4
)
opt_disc = optim.AdamW(discriminator.parameters(), lr=LR*0.5, weight_decay=1e-4)

sched_enc_dec = optim.lr_scheduler.CosineAnnealingLR(opt_enc_dec, T_max=EPOCHS, eta_min=LR_MIN)
sched_disc    = optim.lr_scheduler.CosineAnnealingLR(opt_disc,    T_max=EPOCHS, eta_min=LR_MIN)
print('Optimisers ready ✅')


Optimisers ready ✅


## 💾 Cell 15 — Checkpoint Utilities

In [14]:
# Only initialise if not already set (survives cell re-runs)
if 'history' not in dir() or history is None:
    history     = {'epoch':[], 'msg_loss':[], 'img_loss':[], 'acc':[], 'psnr':[]}
if 'best_acc' not in dir():
    best_acc    = 0.0
if 'start_epoch' not in dir():
    start_epoch = 0


def save_checkpoint(path, epoch, is_best=False):
    torch.save({
        'epoch'         : epoch,
        'encoder'       : encoder.state_dict(),
        'decoder'       : decoder.state_dict(),
        'discriminator' : discriminator.state_dict(),
        'opt_enc_dec'   : opt_enc_dec.state_dict(),
        'opt_disc'      : opt_disc.state_dict(),
        'sched_enc_dec' : sched_enc_dec.state_dict(),
        'sched_disc'    : sched_disc.state_dict(),
        'history'       : history,
        'best_acc'      : best_acc,
        'use_bch'       : USE_BCH,
        'msg_len'       : MESSAGE_LENGTH,
    }, path)
    tag = '🏆 BEST' if is_best else '💾 LAST'
    print(f'  {tag} checkpoint → {os.path.basename(path)}')


def load_checkpoint(path):
    global best_acc, start_epoch, history

    if not os.path.exists(path):
        print(f'  No checkpoint found at {path}. Starting fresh.')
        return False

    ckpt = torch.load(path, map_location=device)

    # ── Compatibility check — reject mismatched arch ──────────────────
    if ckpt.get('msg_len') != MESSAGE_LENGTH:
        print(f'  ⚠️  msg_len mismatch ({ckpt.get("msg_len")} vs {MESSAGE_LENGTH}). Deleting stale checkpoint.')
        os.remove(path)
        return False
    if ckpt.get('use_bch') != USE_BCH:
        print(f'  ⚠️  USE_BCH mismatch ({ckpt.get("use_bch")} vs {USE_BCH}). Deleting stale checkpoint.')
        os.remove(path)
        return False

    # ── Load models ───────────────────────────────────────────────────
    try:
        encoder.load_state_dict(ckpt['encoder'])
        decoder.load_state_dict(ckpt['decoder'])
        discriminator.load_state_dict(ckpt['discriminator'])
    except RuntimeError as e:
        print(f'  ⚠️  Model weight mismatch — arch changed. Deleting stale checkpoint.')
        print(f'      {str(e)[:120]}')
        os.remove(path)
        return False

    # ── Load optimisers & schedulers ──────────────────────────────────
    opt_enc_dec.load_state_dict(ckpt['opt_enc_dec'])
    opt_disc.load_state_dict(ckpt['opt_disc'])
    sched_enc_dec.load_state_dict(ckpt['sched_enc_dec'])
    sched_disc.load_state_dict(ckpt['sched_disc'])

    # ── Restore training state ────────────────────────────────────────
    history.clear()
    history.update(ckpt['history'])
    best_acc    = ckpt['best_acc']
    start_epoch = ckpt['epoch'] + 1

    print(f'  ✅ Resumed from epoch {ckpt["epoch"]}  '
          f'(best acc={best_acc*100:.2f}%  next epoch={start_epoch})')
    return True


# ── Auto-resume from last checkpoint ─────────────────────────────────
load_checkpoint(LAST_CKPT)

  ✅ Resumed from epoch 62  (best acc=87.79%  next epoch=63)


True

## 🖼️ Cell 16 — Collage Logger (Drive)

In [15]:
def bits_to_hex(bits_tensor):
    bits = (bits_tensor > 0.5).int().cpu().numpy().astype(np.uint8)
    pad  = (8 - len(bits)%8) % 8
    bits = np.concatenate([bits, np.zeros(pad, dtype=np.uint8)])
    return ''.join(f'{b:02x}' for b in np.packbits(bits))


def save_collage(cover, encoded, msg, decoded_msg, epoch, batch_idx, mean_acc):
    """
    Saves a 4-sample visual collage to Google Drive.
    Columns: Cover | Watermarked | Difference×10
    Footer : MSG hex | Decoded hex | per-sample bit accuracy
    """
    n = min(4, cover.size(0))
    fig, axes = plt.subplots(n, 3, figsize=(12, n*4))
    if n == 1: axes = axes[np.newaxis, :]
    for i in range(n):
        cov  = np.clip(cover[i].cpu().permute(1,2,0).numpy(), 0, 1)
        enc  = np.clip(encoded[i].detach().cpu().permute(1,2,0).numpy(), 0, 1)
        diff = np.clip((enc - cov)*10 + 0.5, 0, 1)

        m_hex = bits_to_hex(msg[i])
        d_hex = bits_to_hex(decoded_msg[i])
        acc_i = ((decoded_msg[i] > 0.5).float() == msg[i]).float().mean().item()
        p_db  = psnr(cover[i:i+1], encoded[i:i+1])

        for j, (im, title) in enumerate([
            (cov,  f'Cover #{i}'),
            (enc,  f'Watermarked  PSNR={p_db:.1f}dB'),
            (diff, 'Difference x10')
        ]):
            axes[i,j].imshow(im)
            axes[i,j].set_title(title, fontsize=8, pad=3)
            axes[i,j].axis('off')

        color = 'green' if acc_i > 0.95 else 'darkorange' if acc_i > 0.85 else 'red'
        axes[i,0].set_xlabel(f'MSG: 0x{m_hex[:16]}...', fontsize=7, color='navy')
        axes[i,2].set_xlabel(f'DEC: 0x{d_hex[:16]}...  Acc={acc_i:.3f}',
                             fontsize=7, color=color)

    fig.suptitle(
        f'Epoch {epoch} | Batch {batch_idx} | Mean Acc={mean_acc:.4f} | BCH={USE_BCH}',
        fontsize=10, fontweight='bold'
    )
    plt.tight_layout()
    fname = os.path.join(LOG_DIR, f'collage_ep{epoch:03d}_b{batch_idx:05d}.png')
    plt.savefig(fname, dpi=80, bbox_inches='tight')
    plt.close(fig)
    print(f'  🖼️  Collage → {os.path.basename(fname)}')


print('Collage logger ready ✅')


Collage logger ready ✅


## 🏋️ Cell 17 — Training Loop

In [16]:
# Paste this after Cell 9 to verify gradients will flow through Swin
for name, p in encoder.swin.named_parameters():
    p.requires_grad = True   # unfreeze Swin if timm froze it by default
print("Swin unfrozen ✅")

Swin unfrozen ✅


In [ ]:
def progress_bar(current, total, width=28):
    filled = int(width * current / max(total,1))
    return f'[{"█"*filled}{"░"*(width-filled)}] {current}/{total}'


print('=' * 65)
print(f'  Epochs={EPOCHS}  IMG={IMG_SIZE}x{IMG_SIZE}  MSG={MESSAGE_LENGTH}b  BCH={USE_BCH}')
print('=' * 65)

global_step = 0

for epoch in range(start_epoch, EPOCHS):
    encoder.train(); decoder.train(); discriminator.train(); noise_layer.train()

    ep_msg = ep_img = ep_acc = ep_psnr = 0.0
    nb = 0
    t0 = time.time()

    for batch_idx, cover in enumerate(train_loader):
        cover = cover.to(device, non_blocking=True)
        B     = cover.size(0)

        # ── 1. Message ────────────────────────────────────────────
        raw_msg = torch.randint(0, 2, (B, DATA_BITS), device=device).float()
        msg = bch_encode_bits(raw_msg) if USE_BCH else raw_msg

        # ── 2. Encoder + Decoder ──────────────────────────────────
        opt_enc_dec.zero_grad()
        encoded = encoder(cover, msg)
        noised  = noise_layer(encoded)
        decoded = decoder(noised)

        # ── 3. Generator loss ─────────────────────────────────────
        loss_msg  = bce_loss(decoded, msg)
        loss_img  = mse_loss(encoded, cover) + ssim_loss(encoded, cover)
        
        # ── Residual clamp — prevents encoder collapsing watermark to zero ── Shrine 
        residual_strength = (encoded - cover).abs().mean()
        loss_img = loss_img + 0.5 * F.relu(0.01 - residual_strength)
        
        
        loss_freq = freq_loss(cover, encoded)
        disc_fake = discriminator(encoded)
        loss_adv  = adv_bce(disc_fake, torch.zeros_like(disc_fake))

        total_g = (LAMBDA_MSG*loss_msg + LAMBDA_IMG*loss_img
                   + LAMBDA_FREQ*loss_freq + LAMBDA_ADV*loss_adv)
        total_g.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(decoder.parameters()), 1.0)
        opt_enc_dec.step()

        # ── 4. Discriminator ──────────────────────────────────────
        opt_disc.zero_grad()
        loss_d = 0.5*(adv_bce(discriminator(cover), torch.zeros_like(discriminator(cover))) +
                      adv_bce(discriminator(encoded.detach()), torch.ones_like(disc_fake)))
        loss_d.backward()
        opt_disc.step()

        # ── 5. Metrics ────────────────────────────────────────────
        with torch.no_grad():
            acc_b  = bit_accuracy(decoded, msg)
            psnr_b = psnr(cover, encoded)
        ep_msg += loss_msg.item(); ep_img += loss_img.item()
        ep_acc += acc_b; ep_psnr += psnr_b
        nb += 1; global_step += 1

        # ── 6. Colab console output ────────────────────────────────
        if batch_idx % PRINT_EVERY == 0:
            bar = progress_bar(batch_idx+1, len(train_loader))
            print(f'Ep {epoch:02d}/{EPOCHS-1} {bar}  '
                  f'MsgL={loss_msg.item():.4f}  '
                  f'ImgL={loss_img.item():.4f}  '
                  f'Acc={acc_b*100:.1f}%  '
                  f'PSNR={psnr_b:.1f}dB  '
                  f't={time.time()-t0:.0f}s')

    # ── End of epoch ────────────────────────────────────────────────
    
    
    
    sched_enc_dec.step(); sched_disc.step()

    avg_msg  = ep_msg / nb
    avg_img  = ep_img / nb
    avg_acc  = ep_acc / nb
    avg_psnr = ep_psnr / nb
    
    

    history['epoch'].append(epoch)
    history['msg_loss'].append(avg_msg)
    history['img_loss'].append(avg_img)
    history['acc'].append(avg_acc)
    history['psnr'].append(avg_psnr)
    
    # ── Save final results into the log ──────────────────────────────
    _log = json.load(open(RUNS_LOG))
    _log[-1]["result"] = {
        "best_acc"      : round(best_acc * 100, 2),
        "last_epoch_acc": round(avg_acc * 100, 2), 
        "final_psnr"    : round(avg_psnr, 2),
        "epochs_trained": epoch + 1,
    }
    with open(RUNS_LOG, 'w') as f:
        json.dump(_log, f, indent=2)
    
    # ── Collage (after averages are ready) ───────────────────────
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        enc_log = encoder(cover[:4], msg[:4])
        dec_log = decoder(enc_log)
    save_collage(cover[:4], enc_log, msg[:4], dec_log,
                 epoch, len(train_loader)-1, avg_acc)
    encoder.train(); decoder.train()

    print('─' * 65)
    print(f'📊 Epoch {epoch:02d}  '
          f'MsgL={avg_msg:.4f}  ImgL={avg_img:.4f}  '
          f'Acc={avg_acc*100:.2f}%  PSNR={avg_psnr:.2f}dB')

    # Always overwrite last checkpoint (yep random selection bhacha noise ko tara random choose garda ni thikai hola 1 file only)
    save_checkpoint(LAST_CKPT, epoch, is_best=False)

    # Overwrite best checkpoint only when accuracy improves (1 file only)
    if avg_acc > best_acc:
        best_acc = avg_acc
        save_checkpoint(BEST_CKPT, epoch, is_best=True)
        print(f'  🎯 New best: {best_acc*100:.2f}%')
    print('─' * 65)

print('\n✅ Training complete!')
print(f'Best bit accuracy achieved: {best_acc*100:.2f}%')


print(f'Run #{_run_id} results saved ✅')


## 📈 Cell 18 — Training History Plots

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))

axes[0].plot(history['epoch'], history['msg_loss'], 'b-o', ms=3)
axes[0].set_title('Message Loss (BCE)'); axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.4)

axes[1].plot(history['epoch'], [a*100 for a in history['acc']], 'g-o', ms=3)
axes[1].axhline(100, color='r', ls='--', alpha=0.4, label='100%')
axes[1].set_title('Bit Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.4)

axes[2].plot(history['epoch'], history['psnr'], 'm-o', ms=3)
axes[2].axhline(40, color='g', ls='--', alpha=0.4, label='40dB')
axes[2].set_title('PSNR (dB)'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.4)

plt.suptitle('Swin Watermark — Training History', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=100)
plt.show()
print('Plot saved to Drive ✅')


Plot saved to Drive ✅


/tmp/ipykernel_47485/2783800828.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🔬 Cell 19 — Inference Demo

In [18]:
# ── Load best model ──────────────────────────────────────────────────────
load_checkpoint(BEST_CKPT)
encoder.eval(); decoder.eval()

# ── Pick a DIV2K test image ───────────────────────────────────────────────
sample_path = next(
    os.path.join(DIV2K_ROOT, f)
    for f in sorted(os.listdir(DIV2K_ROOT))
    if f.lower().endswith('.png')
)
to_t  = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
cover = to_t(Image.open(sample_path).convert('RGB')).unsqueeze(0).to(device)

# ── Test message ──────────────────────────────────────────────────────────
raw = torch.randint(0, 2, (1, DATA_BITS), device=device).float()
msg = bch_encode_bits(raw) if USE_BCH else raw

# ── Encode → Decode ───────────────────────────────────────────────────────
with torch.no_grad():
    encoded = encoder(cover, msg)
    decoded = decoder(encoded)

acc  = bit_accuracy(decoded, msg)
p_db = psnr(cover, encoded)

if USE_BCH:
    recovered = bch_decode_bits(decoded)
    raw_acc   = bit_accuracy(recovered, raw)
    print(f'BCH corrected raw bit accuracy: {raw_acc*100:.2f}%')

print(f'Codeword bit accuracy : {acc*100:.2f}%')
print(f'PSNR                  : {p_db:.2f} dB')
print(f'Original  (hex) : 0x{bits_to_hex(msg[0])}')
print(f'Recovered (hex) : 0x{bits_to_hex(decoded[0])}')

# ── Visualise ─────────────────────────────────────────────────────────────
diff = (encoded - cover).abs() * 10
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, title in zip(axes,
    [cover[0], encoded[0], diff[0]],
    ['Cover', f'Watermarked (PSNR {p_db:.1f}dB)', 'Diff x10']):
    ax.imshow(np.clip(im.cpu().permute(1,2,0).numpy(), 0, 1))
    ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle(f'Bit Accuracy: {acc*100:.2f}%  |  BCH: {USE_BCH}', fontweight='bold')
plt.tight_layout()
plt.show()


  ✅ Resumed from epoch 29  (best acc=87.79%  next epoch=30)
Codeword bit accuracy : 89.06%
PSNR                  : 33.97 dB
Original  (hex) : 0xa91bf0c54f9c0eff
Recovered (hex) : 0xa71bf0d34f9c0aff


/tmp/ipykernel_47485/329662516.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🔁 Cell 20 — BCH Toggle Guide

In [ ]:
# ── BCH mode switching guide ─────────────────────────────────────────────
#
#  BCH OFF (default):
#    DATA_BITS = MESSAGE_LENGTH = 64
#    No encoding overhead. Faster training. Good starting point.
#
#  BCH ON (USE_BCH = True):
#    DATA_BITS = 64  →  BCH encodes to MESSAGE_LENGTH ≈ 104-127 bits
#    BCH(t=10) corrects up to 10 bit-flips in the decoded codeword.
#    Codeword length is printed when USE_BCH=True is set in Cell 4.
#
#  To switch:
#    1. Change USE_BCH in Cell 4, re-run Cell 4
#    2. Re-run Cell 5  (BCH helpers)
#    3. Re-run Cell 9  (encoder — MESSAGE_LENGTH changes)
#    4. Re-run Cell 10 (decoder — MESSAGE_LENGTH changes)
#    5. Re-run Cell 14 (new optimisers)
#    6. Re-run Cell 15 (fresh history)
#    7. Re-run Cell 17 (training)

print(f'Current mode: USE_BCH={USE_BCH}  MESSAGE_LENGTH={MESSAGE_LENGTH}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SINGLE IMAGE ATTACK TEST — Photometric + Geometric
#  Attack ranges match paper tables (VIII, IX, VI)
# ══════════════════════════════════════════════════════════════════
import io, cv2, numpy as np

encoder.cpu(); decoder.cpu()
test_device = torch.device('cpu')

load_checkpoint(LAST_CKPT)
encoder.eval(); decoder.eval()

# ── Pick image ────────────────────────────────────────────────────
sample_path = "/home/shrine/Minor/RoWSFormer/data/DIV2K/DIV2K_train_HR/DIV2K_train_HR/0003.png"

to_t  = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
cover = to_t(Image.open(sample_path).convert('RGB')).unsqueeze(0).to(test_device)

torch.manual_seed(42)
raw_msg = torch.randint(0, 2, (1, DATA_BITS), device=test_device).float()
msg     = bch_encode_bits(raw_msg) if USE_BCH else raw_msg

with torch.no_grad():
    encoded = encoder(cover, msg)

# ── Helpers ───────────────────────────────────────────────────────
def to_numpy(t):
    return (t[0].detach().cpu().permute(1,2,0).clamp(0,1).numpy() * 255).astype(np.uint8)

def to_tensor(arr, device):
    return torch.from_numpy(arr).permute(2,0,1).float().unsqueeze(0).to(device) / 255.0

def jpeg_approx(x, quality=75):
    imgs = []
    for img in x.cpu():
        pil = TF.to_pil_image(img.clamp(0, 1))
        buf = io.BytesIO()
        pil.save(buf, format='JPEG', quality=quality)
        buf.seek(0)
        imgs.append(TF.to_tensor(Image.open(buf).convert('RGB')))
    return torch.stack(imgs).to(x.device)

def cv_rotate(x, angle):
    arr = to_numpy(x)
    H, W = arr.shape[:2]
    M   = cv2.getRotationMatrix2D((W//2, H//2), angle, 1.0)
    out = cv2.warpAffine(arr, M, (W, H), borderMode=cv2.BORDER_REFLECT)
    return to_tensor(out, x.device)

def cv_scale(x, scale):
    arr = to_numpy(x)
    H, W = arr.shape[:2]
    new_h, new_w = int(H * scale), int(W * scale)
    scaled = cv2.resize(arr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    if scale > 1.0:
        top  = (new_h - H) // 2
        left = (new_w - W) // 2
        out  = scaled[top:top+H, left:left+W]
    else:
        pad_h = (H - new_h) // 2
        pad_w = (W - new_w) // 2
        out = cv2.copyMakeBorder(scaled, pad_h, H-new_h-pad_h,
                                 pad_w, W-new_w-pad_w, cv2.BORDER_REFLECT)
    return to_tensor(out[:H, :W], x.device)

def cv_cropout(x, ratio):
    """Zero out a random patch of given ratio."""
    t   = x.clone()
    B, C, H, W = t.shape
    ch, cw = int(H * ratio), int(W * ratio)
    top  = random.randint(0, H - ch)
    left = random.randint(0, W - cw)
    t[:, :, top:top+ch, left:left+cw] = 0.0
    return t

# ── Attack dictionaries — ranges from paper ───────────────────────
photometric_attacks = {
    "No attack"            : lambda x: x,
    "Gaussian noise σ=0.02": lambda x: torch.clamp(x + 0.02*torch.randn_like(x), 0, 1),
    "Gaussian noise σ=0.05": lambda x: torch.clamp(x + 0.05*torch.randn_like(x), 0, 1),
    "Color jitter (0.1)"   : lambda x: T.ColorJitter(brightness=0.1, contrast=0.1)(x),
    "Gaussian blur k=3"    : lambda x: T.GaussianBlur(3, sigma=(0.5, 1.0))(x),
    "JPEG quality=75"      : lambda x: jpeg_approx(x, quality=75),
    "JPEG quality=80"      : lambda x: jpeg_approx(x, quality=80),
}

# Table VIII — rotation angles tested in paper
rotation_attacks = {
        "Rotate -30°"          : lambda x: cv_rotate(x, -30),
    "Rotate -15°"          : lambda x: cv_rotate(x, -15),
    "Rotate 0° (identity)" : lambda x: x,
    "Rotate +15°"          : lambda x: cv_rotate(x, 15),
    "Rotate +30°"          : lambda x: cv_rotate(x, 30),
}

# Table IX — scaling ratios tested in paper
scale_attacks = {
    "Scale r=0.5"          : lambda x: cv_scale(x, 0.5),
    "Scale r=0.7"          : lambda x: cv_scale(x, 0.7),
    "Scale r=1.0 (identity)": lambda x: x,
    "Scale r=1.5"          : lambda x: cv_scale(x, 1.5),
    "Scale r=2.0"          : lambda x: cv_scale(x, 2.0),
}

# Table VI — cropout ratios tested in paper
cropout_attacks = {
    "Cropout r=0.1"        : lambda x: cv_cropout(x, 0.1),
    "Cropout r=0.2"        : lambda x: cv_cropout(x, 0.2),
    "Cropout r=0.3"        : lambda x: cv_cropout(x, 0.3),
    "Cropout r=0.4"        : lambda x: cv_cropout(x, 0.4),
    "Cropout r=0.5"        : lambda x: cv_cropout(x, 0.5),
}

# ── Run attacks ───────────────────────────────────────────────────
def run_attacks(attack_dict, encoded, decoder, msg, label):
    print(f'\n── {label} ──────────────────────────────')
    print(f'{"Attack":<26} {"Bit Acc":>8} {"Status":>4}')
    print('─' * 45)
    results = {}
    with torch.no_grad():
        for name, fn in attack_dict.items():
            noised  = fn(encoded)
            decoded = decoder(noised)
            acc     = bit_accuracy(decoded, msg)
            results[name] = acc
            marker  = "✅" if acc > 0.85 else ("⚠️" if acc > 0.75 else "❌")
            print(f'{name:<26} {acc*100:>7.2f}%   {marker}')
    return results

print(f'Image : {os.path.basename(sample_path)}')
print(f'PSNR  : {psnr(cover, encoded):.2f} dB\n')

photo_results   = run_attacks(photometric_attacks, encoded, decoder, msg, "Photometric Attacks")
rot_results     = run_attacks(rotation_attacks,    encoded, decoder, msg, "Rotation Attacks    (Table VIII)")
scale_results   = run_attacks(scale_attacks,       encoded, decoder, msg, "Scaling Attacks     (Table IX)")
cropout_results = run_attacks(cropout_attacks,     encoded, decoder, msg, "Cropout Attacks     (Table VI)")

# ── Visual grid per attack group ──────────────────────────────────
def save_grid(attack_dict, results, cover, title, filename):
    n = len(attack_dict)
    fig, axes = plt.subplots(1, n + 1, figsize=(4*(n+1), 4))
    axes[0].imshow(cover[0].permute(1,2,0).clamp(0,1).numpy())
    axes[0].set_title('Cover', fontsize=9); axes[0].axis('off')
    with torch.no_grad():
        for ax, (name, fn) in zip(axes[1:], attack_dict.items()):
            noised = fn(encoded)
            ax.imshow(noised[0].permute(1,2,0).clamp(0,1).numpy())
            ax.set_title(f'{name}\n{results[name]*100:.1f}%', fontsize=8)
            ax.axis('off')
    plt.suptitle(title, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(LOG_DIR, filename)
    plt.savefig(out, bbox_inches='tight', dpi=150)
    plt.close()
    print(f'Saved → {out}')

save_grid(rotation_attacks,    rot_results,     cover, 'Rotation Attacks',  'test_rotation.png')
save_grid(scale_attacks,       scale_results,   cover, 'Scaling Attacks',   'test_scaling.png')
save_grid(cropout_attacks,     cropout_results, cover, 'Cropout Attacks',   'test_cropout.png')
save_grid(photometric_attacks, photo_results,   cover, 'Photometric Attacks','test_photometric.png')

# ── Restore to GPU ────────────────────────────────────────────────
encoder.to(device); decoder.to(device)

  ✅ Resumed from epoch 62  (best acc=87.79%  next epoch=63)
Image : 0003.png
PSNR  : 30.11 dB


── Photometric Attacks ──────────────────────────────
Attack                      Bit Acc Status
─────────────────────────────────────────────
No attack                    85.94%   ✅
Gaussian noise σ=0.02        87.50%   ✅
Gaussian noise σ=0.05        89.06%   ✅
Color jitter (0.1)           85.94%   ✅
Gaussian blur k=3            89.06%   ✅
JPEG quality=75              78.12%   ⚠️
JPEG quality=80              78.12%   ⚠️

── Rotation Attacks    (Table VIII) ──────────────────────────────
Attack                      Bit Acc Status
─────────────────────────────────────────────
Rotate -30°                  59.38%   ❌
Rotate -15°                  78.12%   ⚠️
Rotate 0° (identity)         85.94%   ✅
Rotate +15°                  76.56%   ⚠️
Rotate +30°                  50.00%   ❌

── Scaling Attacks     (Table IX) ──────────────────────────────
Attack                      Bit Acc Status
───────────

ConvDecoder(
  (body): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): FrequencyEnhancement(
      (low): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
        (1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
        (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (3): GELU(approximate='none')
      )
      (high): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
        (1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
        (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (3): GELU(approximate='none')
      )
      (gate): Sequential(
        (0): AdaptiveAvgPool2d(output_size=1)
        (1): Flatten(start_dim